# Columbia ~1km Network: Euclidean vs. Network Distance + Xin et al. (2022)–style Metrics

**Research statement.**  
Within ~1 km around Columbia University (Morningside), we build a street network from NYC street centerlines and quantify how **Euclidean distance** differs from **shortest-path distance** along streets. We then simulate origin–destination flows and compute **macro indicators** inspired by Xin et al. (2022) to interpret connectivity and flow heterogeneity at this local scale.



## Data Visualization Results

The following five key figures were generated from this analysis:

### 1. Columbia University Area Street Network Map
![Columbia Network Map](columbia_network_map.png)

### 2. Euclidean vs Network Distance Scatter Plot
![Euclidean vs Network Distance](real_scatter_euclid_vs_network.png)

### 3. Detour Index Distribution Histogram
![Detour Index Distribution](real_hist_detour.png)

### 4. Network Node Degree Distribution Histogram
![Degree Distribution](real_degree_hist.png)

### 5. Edge Flow Intensity Map
![Edge Flow Map](real_edge_flow_map.png)

These visualizations demonstrate the topological characteristics, distance relationships, and simulated flow distribution of the street network within a 1km radius around Columbia University.

## 1) Data & Network Construction
We parse LineString segments from GeoJSON, snap endpoints to merge near-duplicates, and build a **directed** graph. Edge weights are **geodesic lengths** (meters) via haversine; coordinates remain lon/lat for mapping.


In [ ]:
# Rebuild the graph from the uploaded file (for full reproducibility)
import json, math, networkx as nx, matplotlib.pyplot as plt, numpy as np, pandas as pd


def haversine_m(lon1, lat1, lon2, lat2):
    R = 6371000.0
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dl / 2) ** 2
    )
    return 2 * R * math.atan2(a**0.5, (1 - a) ** 0.5)


def quantize_coord(lon, lat, q=1e-7):
    return (round(lon / q) * q, round(lat / q) * q)


path = "/mnt/data/columbia_1km.geojson"
gj = json.load(open(path, "r", encoding="utf-8"))
lines = [
    ft["geometry"]["coordinates"]
    for ft in gj["features"]
    if ft.get("geometry", {}).get("type") == "LineString"
]

G = nx.DiGraph()
for coords in lines:
    for i in range(len(coords) - 1):
        (lon1, lat1) = coords[i]
        (lon2, lat2) = coords[i + 1]
        a = quantize_coord(lon1, lat1, 1e-7)
        b = quantize_coord(lon2, lat2, 1e-7)
        if a == b:
            continue
        if a not in G:
            G.add_node(a, lon=a[0], lat=a[1])
        if b not in G:
            G.add_node(b, lon=b[0], lat=b[1])
        d = haversine_m(a[0], a[1], b[0], b[1])
        G.add_edge(a, b, length=d)
        G.add_edge(b, a, length=d)

len(G), G.number_of_edges()

### Map with a sample shortest path (Low Library → near 125th St)


In [ ]:
def plot_network_lonlat(G, highlight_path=None):
    for u, v, d in G.edges(data=True):
        x1, y1 = G.nodes[u]["lon"], G.nodes[u]["lat"]
        x2, y2 = G.nodes[v]["lon"], G.nodes[v]["lat"]
        plt.plot([x1, x2], [y1, y2], linewidth=0.5, alpha=0.8)
    if highlight_path:
        for i in range(len(highlight_path) - 1):
            a, b = highlight_path[i], highlight_path[i + 1]
            x1, y1 = G.nodes[a]["lon"], G.nodes[a]["lat"]
            x2, y2 = G.nodes[b]["lon"], G.nodes[b]["lat"]
            plt.plot([x1, x2], [y1, y2], linewidth=3.0)
    plt.scatter(
        [G.nodes[n]["lon"] for n in G.nodes()],
        [G.nodes[n]["lat"] for n in G.nodes()],
        s=6,
    )
    plt.gca().set_aspect("equal", adjustable="box")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")


def nearest_node(G, lon, lat):
    best, bestd = None, float("inf")
    for n in G.nodes():
        d = haversine_m(lon, lat, G.nodes[n]["lon"], G.nodes[n]["lat"])
        if d < bestd:
            best, bestd = n, d
    return best


low_lib = (-73.9626, 40.8075)
harlem_125 = (-73.955, 40.8155)
u_demo = nearest_node(G, *low_lib)
v_demo = nearest_node(G, *harlem_125)

import networkx as nx

try:
    path = nx.shortest_path(G, u_demo, v_demo, weight="length")
except nx.NetworkXNoPath:
    path = None

plt.figure(figsize=(7, 6))
plot_network_lonlat(G, path)
plt.title("Columbia ~1km (sample path)")
plt.show()

## 2) Euclidean vs. Network Distance


In [ ]:
import numpy as np, pandas as pd, math


def euclid_geo_m(a, b):
    return haversine_m(a[0], a[1], b[0], b[1])


pairs = []
nodes = list(G.nodes())
rng = np.random.default_rng(0)
for _ in range(300):
    u, v = rng.choice(nodes, 2, replace=False)
    try:
        dG = nx.shortest_path_length(G, u, v, weight="length")
    except nx.NetworkXNoPath:
        continue
    dE = euclid_geo_m(u, v)
    if dE > 0:
        pairs.append(
            {
                "origin": u,
                "dest": v,
                "euclidean_m": dE,
                "network_m": dG,
                "detour_index": dG / dE,
            }
        )
dist_df = pd.DataFrame(pairs)
dist_df.describe(numeric_only=True)

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(dist_df["euclidean_m"], dist_df["network_m"], s=8, alpha=0.7)
plt.xlabel("Euclidean (m)")
plt.ylabel("Network (m)")
plt.title("Euclidean vs. Network Distance")
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(dist_df["detour_index"], bins=25, alpha=0.9)
plt.xlabel("Detour index (network/Euclid)")
plt.ylabel("Count")
plt.title("Detour Index Distribution")
plt.show()

## 3) OD Simulation + Xin-style Indicators


In [ ]:
def simulate_od_flows(G, n_trips=8000, friction=0.8):
    nodes = list(G.nodes())
    pairs, probs = [], []
    for u in nodes:
        for v in nodes:
            if u == v:
                continue
            dE = haversine_m(
                G.nodes[u]["lon"],
                G.nodes[u]["lat"],
                G.nodes[v]["lon"],
                G.nodes[v]["lat"],
            )
            p = math.exp(-friction * (dE / 1000.0))
            pairs.append((u, v))
            probs.append(p)
    probs = np.array(probs)
    probs = probs / probs.sum()
    idx = np.random.choice(np.arange(len(pairs)), size=n_trips, p=probs)
    sampled = [pairs[i] for i in idx]
    node_flow = {n: 0 for n in nodes}
    edge_flow = {e: 0 for e in G.edges()}
    trips = []
    for u, v in sampled:
        try:
            path = nx.shortest_path(G, u, v, weight="length")
        except nx.NetworkXNoPath:
            continue
        node_flow[u] += 1
        node_flow[v] += 1
        dist = 0.0
        for i in range(len(path) - 1):
            a, b = path[i], path[i + 1]
            edge_flow[(a, b)] += 1
            dist += G[a][b]["length"]
        dE = haversine_m(
            G.nodes[u]["lon"], G.nodes[u]["lat"], G.nodes[v]["lon"], G.nodes[v]["lat"]
        )
        trips.append(
            {
                "origin": u,
                "dest": v,
                "euclidean_m": dE,
                "network_m": dist,
                "detour_index": (dist / dE if dE > 0 else np.nan),
            }
        )
    return node_flow, edge_flow, pd.DataFrame(trips)


def xin_metrics(G, node_flow, edge_flow):
    N = G.number_of_nodes()
    L = G.number_of_edges()
    K_avg = np.mean([G.out_degree(n) + G.in_degree(n) for n in G.nodes()])
    delta = (2 * L) / (N**2)
    C = nx.transitivity(G.to_undirected())
    F_vals = np.array(list(node_flow.values()))
    W_vals = np.array(list(edge_flow.values()))
    return {
        "N": N,
        "L": L,
        "<K>": K_avg,
        "δ": delta,
        "C": C,
        "<F>": float(F_vals.mean()),
        "CV(F)": float(F_vals.std(ddof=0) / (F_vals.mean() + 1e-9)),
        "<W>": float(W_vals.mean()),
        "CV(W)": float(W_vals.std(ddof=0) / (W_vals.mean() + 1e-9)),
    }


node_flow, edge_flow, trips_df = simulate_od_flows(G)
metrics = xin_metrics(G, node_flow, edge_flow)
metrics

In [ ]:
deg_vals = [G.degree(n) for n in G.nodes()]
plt.figure(figsize=(6, 4))
plt.hist(deg_vals, bins=20, alpha=0.9)
plt.xlabel("Degree")
plt.ylabel("Count")
plt.title("Degree Distribution (Columbia ~1km)")
plt.show()

In [ ]:
flows = np.array(list(edge_flow.values()))
maxf = flows.max() if len(flows) else 1.0
plt.figure(figsize=(7, 6))
for a, b in G.edges():
    x1, y1 = G.nodes[a]["lon"], G.nodes[a]["lat"]
    x2, y2 = G.nodes[b]["lon"], G.nodes[b]["lat"]
    lw = 0.4 + 3.0 * (edge_flow[(a, b)] / maxf) if maxf > 0 else 0.5
    plt.plot([x1, x2], [y1, y2], linewidth=lw, alpha=0.9)
plt.scatter(
    [G.nodes[n]["lon"] for n in G.nodes()], [G.nodes[n]["lat"] for n in G.nodes()], s=6
)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.gca().set_aspect("equal", adjustable="box")
plt.title("Edge Flow Intensity (linewidth ∝ flow)")
plt.show()